# pandas: filtration

Filtering is the process of selecting a subset of your data based on criteria that you specify.

In this lesson I'll demonstrate how to leverage the `Series.filter()`, `DataFrame.filter()` and `DataFrameGroupBy.filter()` methods as we continue to explore the Mega Millions lottery data covering the period 2017-2024.

💡 I've adjusted the `IPython` interactive shell's `ast_node_interactivity` setting to ensure that all cell output is displayed. This overrides the default last expression only ("last_expr") display behavior.

In [7040]:
import IPython as ipy
import pandas as pd


# Ensure that all interactive cell output is displayed (default="last_expr")
ipy.get_ipython().ast_node_interactivity = "all"

## Retrieve and prep the data

Let's read in the winning numbers data, convert the "draw_date" column dtype from `object` to `datetime64[ns]`, review a summary of the new `DataFrame` named `data`, and inspect its first three rows.

In [7041]:
data = pd.read_csv("./data/mega_millions-aggregate-2017_24.csv")
data["draw_date"] = pd.to_datetime(data["draw_date"], format="%Y-%m-%d")
data.info()
data.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   draw_date        684 non-null    datetime64[ns]
 1   winning_numbers  684 non-null    object        
 2   mega_ball        684 non-null    int64         
 3   multiplier       684 non-null    int64         
 4   pick5_1          684 non-null    int64         
 5   pick5_2          684 non-null    int64         
 6   pick5_3          684 non-null    int64         
 7   pick5_4          684 non-null    int64         
 8   pick5_5          684 non-null    int64         
 9   pick5_sum        684 non-null    int64         
 10  mega_ball_mod2   684 non-null    int64         
 11  pick5_mod2_sum   684 non-null    int64         
dtypes: datetime64[ns](1), int64(10), object(1)
memory usage: 64.3+ KB


,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,mega_ball_mod2,pick5_mod2_sum
0,2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70,195,1,1
1,2024-05-14,13 19 43 62 64,6,3,13,19,43,62,64,201,0,3
2,2024-05-10,13 22 26 32 65,18,4,13,22,26,32,65,158,0,2


## Filtration

If you need to return a subset of a `Series` or `DataFrame` based on index labels call either the [`Series.filter()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.filter.html) method or the [`DataFrame.filter()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html) method. The operation applies a filter to the labels of the index rather than the content. The filter can be applied to either the index (`axis=0/"index"`) or the column index (`axis=1/"columns"/None`).

`Series.filter(items=None, like=None, regex=None, axis=None)`

`DataFrame.filter(items=None, like=None, regex=None, axis=None)`

The caller is limited to specifying a _single_ filtering condition using either the `items`, `like`, or `regex` parameters. If multiple conditions are specified using these parameters the method call will trigger a `TypeError`.

```commandline
TypeError: Keyword arguments `items`, `like`, or `regex` are mutually exclusive
```

### Exact match

The following example demonstrates use of the `items` parameter to pass a tuple of label names to the `DataFrame.filter()` method. The filter is applied along the column axis (in this case, `axis=None`) and a new `DataFrame` comprising the five "pick5_*" columns" is returned to the caller.

In [7042]:
labels = ("pick5_1", "pick5_2", "pick5_3", "pick5_4", "pick5_5")
pick5 = data.filter(items=labels)  # axis=None
pick5.head(3)

,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,8,17,40,60,70
1,13,19,43,62,64
2,13,22,26,32,65


Alternatively, you can pass `axis=1` or `axis="columns"`.

In [7043]:
pick5 = data.filter(items=labels, axis=1)
pick5.head(3)

,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,8,17,40,60,70
1,13,19,43,62,64
2,13,22,26,32,65


### Partial match

If you need to filter on one or more index labels that are named similarly utilize the `like` parameter. In the next example the filter is applied along the column axis. All columns whose labels _contain_ the substring "pick5_" are retained and returned to the caller in a new `DataFrame`.

In [7044]:
pick5 = data.filter(like="pick5_", axis="columns")
pick5.tail(3)

,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,pick5_mod2_sum
681,1,54,60,68,69,252,2
682,10,22,42,61,69,204,2
683,6,28,31,52,53,170,2


### Pattern match

For more nuanced filtering consider utilizing the `regex` parameter to filter on a regular expression. The "regex" filtering expression below targets the Pick 5 winning number columns; all other columns whose labels contain the substring "pick5_" are ignored.

In [7045]:
pick5 = data.filter(regex="pick5_[1-5]", axis=1)
pick5.head(3)

,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,8,17,40,60,70
1,13,19,43,62,64
2,13,22,26,32,65


### Row filtering

You can also filter on the index by specifying `axis=0`.

In the example below, I've taken the liberty of calling the [`Dataframe.set_index()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) method and returning a new `DataFrame` named `draws` that features an index based on the "draw_date" column. I then call the `DataFrame.filter()` method, passing it a list of datetime indices upon which to filter along with the all important keyword argument `axis=0` to return the desired rows.

In [7046]:
# Specified dates
indices = [
    pd.to_datetime(draw_date)
    for draw_date in ("2024-05-17", "2023-05-16", "2022-05-17", "2021-05-14", "2020-05-15")
]

# Random dates
# rng = np.random.default_rng(506)  # random number generator
# indices = data.iloc[rng.integers(0, len(data), 5)]["draw_date"]

draws = data.set_index("draw_date").sort_index(ascending=False)  # set up
rows = draws.filter(items=indices, axis=0)  # axis="index"
rows

,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,mega_ball_mod2,pick5_mod2_sum
2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70,195,1,1
2023-05-16,15 34 36 69 70,17,3,15,34,36,69,70,224,1,2
2022-05-17,07 21 24 41 65,24,4,7,21,24,41,65,158,0,4
2021-05-14,03 18 41 44 68,3,2,3,18,41,44,68,174,1,2
2020-05-15,11 17 32 33 46,25,3,11,17,32,33,46,139,1,3


## Group-wise filtering

Likewise, the `DataFrameGroupBy` object includes a `filter()` method for removing elements from groups that do not meet the filtering criteria. After creating a `DataFrameGroupBy` object you can call the `filter()` method and pass it a function (either named or anonymous) that evaluates each group for inclusion/exclusion per a specified condition.

In the example below method chaining is employed to group `data` by the "pick5_mod2_sum" column and then filter the resulting groups by counting the number of rows that match the filter expression. The row count is then compared to a threshold value of `200`. If the count exceeds `200` the boolean `True` is returned to the caller; otherwise `False` is returned. Groups with less than 200 rows are dropped from the `DataFrame` returned by calling `DataFrameGroupBy.filter()`.

Note that the expression passed to the `filter()` method _must_ evaluate to `True` or `False`, otherwise a runtime exception will occur. For example, neglecting to **count** the individual rows will trigger a `TypeError` due to the `filter()` method returning a `Series` object rather than a scalar boolean.

```python
data.groupby("pick5_mod2_sum").filter(lambda x: x["pick5_mod2_sum"] > 200)  # Triggers a TypeError
```

In [7047]:
# Count rows not the sum of associated values; returns DataFrame
groups_gt_200 = data.groupby("pick5_mod2_sum").filter(lambda x: x["pick5_mod2_sum"].count() > 200)
groups_gt_200

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,mega_ball_mod2,pick5_mod2_sum
1,2024-05-14,13 19 43 62 64,6,3,13,19,43,62,64,201,0,3
2,2024-05-10,13 22 26 32 65,18,4,13,22,26,32,65,158,0,2
4,2024-05-03,06 13 15 53 56,11,2,6,13,15,53,56,143,1,3
5,2024-04-30,10 18 27 37 61,5,3,10,18,27,37,61,153,1,3
9,2024-04-16,21 26 36 44 59,2,4,21,26,36,44,59,186,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...
678,2017-11-17,03 26 55 58 70,15,4,3,26,55,58,70,212,1,2
679,2017-11-14,01 14 21 22 28,19,3,1,14,21,22,28,86,1,2
681,2017-11-07,01 54 60 68 69,11,4,1,54,60,68,69,252,1,2
682,2017-11-03,10 22 42 61 69,3,2,10,22,42,61,69,204,1,2


The second example filters out groups that fall on either side of one standard deviation of the "pick5_sum" column mean.

In [7048]:
mu = data["pick5_sum"].mean()
sigma = data["pick5_sum"].std()

print(f"mu={mu} ", f"sigma={sigma} ", f"below={mu - sigma} ", f"above={mu + sigma}\n")

data_sigma_one = data.groupby("pick5_sum").filter(
    lambda x: x["pick5_sum"].between(mu - sigma, mu + sigma).all()
)
data_sigma_one

mu=173.23976608187135  sigma=43.94563855956432  below=129.29412752230704  above=217.18540464143567



,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,pick5_sum,mega_ball_mod2,pick5_mod2_sum
0,2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70,195,1,1
1,2024-05-14,13 19 43 62 64,6,3,13,19,43,62,64,201,0,3
2,2024-05-10,13 22 26 32 65,18,4,13,22,26,32,65,158,0,2
4,2024-05-03,06 13 15 53 56,11,2,6,13,15,53,56,143,1,3
5,2024-04-30,10 18 27 37 61,5,3,10,18,27,37,61,153,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...
675,2017-11-28,10 17 47 51 61,5,2,10,17,47,51,61,186,1,4
678,2017-11-17,03 26 55 58 70,15,4,3,26,55,58,70,212,1,2
680,2017-11-10,06 23 38 42 58,24,2,6,23,38,42,58,167,0,1
682,2017-11-03,10 22 42 61 69,3,2,10,22,42,61,69,204,1,2


### Write to file

Let's conclude our discussion of `Series`, `DataFrame`, and `DataFrameGroupBy` filtering by writing `data_sigma_one` to a CSV file. We may make use of the data file in a future lesson. 🙂

In [7049]:
filepath = "./data/mega_millions-filter-sigma_one-2017_24.csv"
data_sigma_one.to_csv(filepath, index=False)